# <center>**Configuration**</center>

In [2]:
# ============================================================
# 1. Private Project Configuration
# ============================================================

import os
from pathlib import Path

from dotenv import load_dotenv


CONFIG_FILE = Path.home() / ".wadidegla.env"

if CONFIG_FILE.exists():
    load_dotenv(CONFIG_FILE, override=False)


data_dir_value = os.getenv("WADIDEGLA_DATA_DIR")
workspace_dir_value = os.getenv("WADIDEGLA_WORKSPACE_DIR")


missing_variables = []

if not data_dir_value:
    missing_variables.append("WADIDEGLA_DATA_DIR")

if not workspace_dir_value:
    missing_variables.append("WADIDEGLA_WORKSPACE_DIR")


if missing_variables:
    raise EnvironmentError(
        "Missing WadiDegla configuration variable(s): "
        + ", ".join(missing_variables)
        + ". Define them in ~/.wadidegla.env."
    )


DATA_DIR = Path(data_dir_value).expanduser()
WORKSPACE_DIR = Path(workspace_dir_value).expanduser()

TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Validation"

PUBLICATION_RECORDS_DIR = (
    WORKSPACE_DIR
    / "WadiDegla_Publication_Records"
)


directories_to_check = {
    "WadiDegla dataset": DATA_DIR,
    "WadiDegla training dataset": TRAIN_DIR,
    "WadiDegla validation dataset": VAL_DIR,
    "WadiDegla experimental workspace": WORKSPACE_DIR,
}


for name, path in directories_to_check.items():
    if not path.exists():
        raise FileNotFoundError(
            f"The configured {name} directory does not exist."
        )


print("✅ WadiDegla project configuration loaded successfully.")

✅ WadiDegla project configuration loaded successfully.


# **<center>Exact species-overlap audit</center>**

In [4]:
# ============================================================
# Audit PlantNet-300K Exact Species Overlap
# Against WadiDegla Target Species
# ============================================================

import json
import re

import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PLANTNET_SPECIES_JSON = (
    WORKSPACE_DIR
    / "plantnet300K_species_names.json"
)

OTHERS_RECORDS_DIR = (
    PUBLICATION_RECORDS_DIR
    / "others_class"
)

OTHERS_RECORDS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


if not PLANTNET_SPECIES_JSON.exists():
    raise FileNotFoundError(
        "plantnet300K_species_names.json was not found "
        "in the configured WadiDegla workspace."
    )


# ============================================================
# 2. Read Current WadiDegla Target Species
# ============================================================

train_classes = {
    path.name
    for path in TRAIN_DIR.iterdir()
    if path.is_dir()
    and not path.name.startswith(".")
}

validation_classes = {
    path.name
    for path in VAL_DIR.iterdir()
    if path.is_dir()
    and not path.name.startswith(".")
}


# Train and Validation should contain identical class labels.
if train_classes != validation_classes:
    raise ValueError(
        "Train and Validation do not contain the same class labels."
    )


# Exclude the aggregate others class.
wadi_target_species = sorted(
    species
    for species in train_classes
    if species.lower() != "others"
)


if len(wadi_target_species) != 32:
    raise ValueError(
        f"Expected 32 WadiDegla target species, "
        f"found {len(wadi_target_species)}."
    )


# ============================================================
# 3. Read PlantNet-300K Species Names
# ============================================================

with open(
    PLANTNET_SPECIES_JSON,
    "r",
    encoding="utf-8"
) as f:
    plantnet_species_dict = json.load(f)


# ============================================================
# 4. Normalize Species Names
# ============================================================

def normalize_species_name(name):
    """
    Convert a scientific name to a normalized binomial:

        Genus + specific epithet

    Authorship is ignored.

    Examples
    --------
    'Peganum_harmala'
        -> 'peganum harmala'

    'Peganum harmala L.'
        -> 'peganum harmala'

    'Cenchrus divisus (J.F.Gmel.) Verloove, Govaerts & Buttler'
        -> 'cenchrus divisus'
    """

    # PlantNet uses underscores instead of spaces.
    name = str(name).replace("_", " ").strip()

    # Normalize repeated whitespace.
    name = re.sub(
        r"\s+",
        " ",
        name
    )

    parts = name.split()

    if len(parts) < 2:
        return name.lower()

    # Exact species identity based on Genus + specific epithet.
    return f"{parts[0]} {parts[1]}".lower()


# ============================================================
# 5. Build WadiDegla Species Table
# ============================================================

wadi_records = pd.DataFrame({
    "wadi_species": wadi_target_species
})


wadi_records["normalized_species"] = (
    wadi_records[
        "wadi_species"
    ]
    .map(normalize_species_name)
)


# ============================================================
# 6. Build PlantNet Species Table
# ============================================================

plantnet_records = pd.DataFrame([
    {
        "plantnet_species_id": species_id,
        "plantnet_species_name": species_name,
    }
    for species_id, species_name
    in plantnet_species_dict.items()
])


plantnet_records["normalized_species"] = (
    plantnet_records[
        "plantnet_species_name"
    ]
    .map(normalize_species_name)
)


# ============================================================
# 7. Find Exact Species-Level Overlap
# ============================================================

exact_overlap = (
    wadi_records
    .merge(
        plantnet_records,
        on="normalized_species",
        how="inner"
    )
    [
        [
            "wadi_species",
            "normalized_species",
            "plantnet_species_id",
            "plantnet_species_name",
        ]
    ]
    .sort_values(
        [
            "wadi_species",
            "plantnet_species_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. Export Audit Results
# ============================================================

exact_overlap.to_csv(
    OTHERS_RECORDS_DIR
    / "plantnet_target_species_overlap.csv",
    index=False
)


audit_summary = {

    "wadi_target_species": len(
        wadi_target_species
    ),

    "plantnet_species_records": len(
        plantnet_records
    ),

    "exact_species_overlaps": len(
        exact_overlap
    ),

    "comparison_level": (
        "Exact species-level comparison only."
    ),

    "comparison_method": (
        "Scientific names were normalized by replacing underscores "
        "with spaces and comparing genus + specific epithet. "
        "Taxonomic authorship was ignored."
    ),

    "synonym_resolution_performed": False,
}


with open(
    OTHERS_RECORDS_DIR
    / "plantnet_target_overlap_audit.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_summary,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 9. Verification Output
# ============================================================

print(
    f"WadiDegla target species: "
    f"{len(wadi_target_species)}"
)

print(
    f"PlantNet species records: "
    f"{len(plantnet_records)}"
)

print(
    f"Exact target-species overlaps: "
    f"{len(exact_overlap)}"
)


if exact_overlap.empty:

    print(
        "✅ No exact WadiDegla target-species overlap "
        "was detected in the PlantNet species list."
    )

else:

    print()
    print(
        "❌ Exact target-species overlap detected:"
    )

    print(
        exact_overlap.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "PlantNet contains one or more exact WadiDegla "
        "target species."
    )

WadiDegla target species: 32
PlantNet species records: 1081
Exact target-species overlaps: 0
✅ No exact WadiDegla target-species overlap was detected in the PlantNet species list.


# <center>**others Class Construction**</center>

The "others" class was constructed using images from the [PlantNet-300K dataset](https://www.kaggle.com/datasets/noahbadoa/plantnet-300k-images).

A pool of 500 images was randomly selected from the PlantNet-300K `images_val`
subset. No explicit random seed was recorded for this initial 500-image
selection.

The original PlantNet source-image directory was subsequently reorganized
and is no longer retained in its original form. Therefore, the exact initial
sampling operation is documented here as a historical procedure rather than
rerun.

The selected 500 images were subsequently partitioned into:
- 400 training images
- 100 validation images

## **Historical Random-Sampling Procedure**

The following function documents the random-sampling procedure used to
construct the initial 500-image pool. It is retained for methodological
documentation only and should not be rerun.

```python
def copy_random_images(base_path, dest_path, num_images):
    """
    Copies a random selection of images from subfolders in base_path
    to dest_path.
    """
    os.makedirs(dest_path, exist_ok=True)

    species_folders = [
        os.path.join(base_path, folder)
        for folder in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, folder))
    ]

    all_images = []

    for folder in species_folders:
        images = [
            os.path.join(folder, img)
            for img in os.listdir(folder)
            if os.path.isfile(os.path.join(folder, img))
        ]

        all_images.extend(images)

    selected_images = random.sample(
        all_images,
        min(num_images, len(all_images))
    )

    for img_path in selected_images:
        shutil.copy(
            img_path,
            os.path.join(
                dest_path,
                os.path.basename(img_path)
            )
        )

    print(
        f"Copied {len(selected_images)} images to {dest_path}."
    )
```

### Historical sampling record

- Source dataset: PlantNet-300K
- Source subset: `images_val`
- Sampling unit: individual image
- Initial number selected: 500 images
- Selection method: random sampling
- Explicit random seed: not recorded
- Original source pool retained in its original form: no
- Original random-selection event exactly reproducible: no
- Selected images currently retained: yes, as the final 400/100 partition
- Final use: 400 training images and 100 validation images for the aggregate
  `others` class

# **<center>Final `others` Dataset Verification</center>**

In [5]:
# ============================================================
# Final `others` Dataset Verification
# ============================================================

import hashlib

import pandas as pd


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
    ".avif",
}


TRAIN_OTHERS_DIR = TRAIN_DIR / "others"
VAL_OTHERS_DIR = VAL_DIR / "others"


for name, directory in {
    "training `others`": TRAIN_OTHERS_DIR,
    "validation `others`": VAL_OTHERS_DIR,
}.items():

    if not directory.is_dir():
        raise FileNotFoundError(
            f"The {name} directory does not exist."
        )


def sha256_file(path, chunk_size=1024 * 1024):
    """
    Compute the SHA-256 hash of a file.
    """

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def collect_others_images(directory, split_name):
    """
    Collect image records recursively from an `others` split.
    """

    records = []

    for image_path in sorted(directory.rglob("*")):

        if (
            image_path.is_file()
            and not image_path.name.startswith(".")
            and image_path.suffix.lower() in IMAGE_EXTENSIONS
        ):

            records.append({
                "split": split_name,

                # Store only a project-relative path.
                "relative_path": str(
                    image_path.relative_to(DATA_DIR)
                ),

                "file_name": image_path.name,

                "sha256": sha256_file(
                    image_path
                ),
            })

    return records


train_records = collect_others_images(
    TRAIN_OTHERS_DIR,
    "Train"
)

validation_records = collect_others_images(
    VAL_OTHERS_DIR,
    "Validation"
)


others_manifest = pd.DataFrame(
    train_records
    + validation_records
)


train_count = len(train_records)
validation_count = len(validation_records)
total_count = len(others_manifest)


print(
    f"Train `others` images: {train_count}"
)

print(
    f"Validation `others` images: {validation_count}"
)

print(
    f"Total `others` images: {total_count}"
)


# ============================================================
# Expected Final Counts
# ============================================================

assert train_count == 400, (
    f"Expected 400 training `others` images, "
    f"found {train_count}."
)

assert validation_count == 100, (
    f"Expected 100 validation `others` images, "
    f"found {validation_count}."
)

assert total_count == 500, (
    f"Expected 500 total `others` images, "
    f"found {total_count}."
)


# ============================================================
# Check Train/Validation Content Overlap
# ============================================================

train_hashes = {
    record["sha256"]
    for record in train_records
}

validation_hashes = {
    record["sha256"]
    for record in validation_records
}


cross_split_duplicates = (
    train_hashes
    & validation_hashes
)


print(
    "Train/Validation duplicate-content hashes:",
    len(cross_split_duplicates)
)


if cross_split_duplicates:

    raise RuntimeError(
        "Duplicate image content was detected between "
        "Train/others and Validation/others."
    )


# ============================================================
# Export Current Final Manifest
# ============================================================

others_manifest.to_csv(
    OTHERS_RECORDS_DIR
    / "others_final_manifest.csv",
    index=False
)


others_verification = {
    "class_name": "others",
    "train_images": train_count,
    "validation_images": validation_count,
    "total_images": total_count,
    "cross_split_duplicate_hashes": len(
        cross_split_duplicates
    ),
}


with open(
    OTHERS_RECORDS_DIR
    / "others_final_verification.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        others_verification,
        f,
        indent=4
    )


print(
    "✅ Final `others` dataset verification completed."
)

Train `others` images: 400
Validation `others` images: 100
Total `others` images: 500
Train/Validation duplicate-content hashes: 0
✅ Final `others` dataset verification completed.
